In [ ]:
import controlimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snssns.set_style("whitegrid")

DC Motor Position: System Analysis

Key MATLAB commands used in this tutorial are:<http://www.mathworks.com/help/toolbox/control/ref/tf.html |tf|> , <http://www.mathworks.com/help/toolbox/control/ref/step.html |step|> , <http://www.mathworks.com/help/toolbox/control/ref/isstable.html |isstable|> , <http://www.mathworks.com/help/toolbox/control/ref/pole.html |pole|> , <http://www.mathworks.com/help/toolbox/control/ref/feedback.html |feedback|> ,<http://www.mathworks.com/help/toolbox/control/ref/pzmap.html |pzmap|> ,<http://www.mathworks.com/help/toolbox/control/ref/damp.html |damp|>

From the main problem, the dynamic equations in the Laplace domain andthe open-loop transfer function of the DC Motor are the following.

$$ s(Js + b)\Theta(s) = KI(s) $$

$$ (Ls + R)I(s) = V(s) - Ks\Theta(s) $$

$$ P(s) = \frac {\Theta(s)}{V(s)} = \frac{K}{s ( (Js + b)(Ls + R) + K^2 )} \qquad [ \frac{rad}{V} ] $$

For the original problem setup and the derivation of the above equations,please refer to the< ?example=MotorPosition&section=SystemModeling DC Motor Position: System Modeling> page. 

For a 1-radian step reference, the design criteria are given are thefollowing.

* Settling time less than 0.040 seconds* Overshoot less than 16%* No steady-state error, even in the presence of a step disturbance input

Open-loop responseFirst create a new < ?aux=Extras_Mfilem-file> and type in the following commands (refer to the main problem forthe details of getting these commands).   

In [1]:
J = 3.2284E-6b = 3.5077E-6K = 0.0274R = 4L = 2.75E-6s = control.tf('s')P_motor = K/(s*((J*s+b)*(L*s+R)+K**2))

Now let's see how the uncompensated open-loop system performs.Specifically, we will use the MATLAB command |step| to analyze theopen-loop step response. Add the following commands onto the end of them-file and run it in the MATLAB command window and you will get theassociated plot shown below. 

In [2]:
t = 0:0.001:0.2T, yout = control.step_response(P_motor,t)plt.show()

From the above plot, we can see that when 1 volt is applied to the systemthe motor position grows unbounded. This is obviously at odds with the givenrequirements, in particular, that there be no steady-state error. Theopen-loop response of the system is not even stable. Stability of asystem can be verified with the MATLAB command |isstable| where areturned value of |TRUE| (1) indicates that the system is stable and areturned value of |FALSE| (0) indicates that the system is not stable.

In [3]:
isstable(P_motor)

Stability of the system can also be determined from the poles of thetransfer function where the poles can be identified using the MATLABcommand |pole| as shown below.

In [4]:
pole(P_motor)

As indicated by this function, one of the poles of the open-loop transferfunction is on the imaginary axis while the other two poles are in theleft half of the complex _s_-plane.  A pole on the imaginary axisindicates that the free response of the system will not grow unbounded,but also will not decay to zero. Even though the free response will not growunbounded, a system with a pole on the imaginary axis can grow unboundedwhen given an input, even when the input is bounded. This fact is inagreement with what we have already seen. In this particular case, thepole at the origin behaves like an integrator. Therefore, when the systemis given a step input its output continues to grow to infinity in thesame manner that an integral of a constant would grow to infinity as theupper limit of the integral is made larger.

Closed-loop responseLet's now consider the closed-loop response of the system where thesystem schematic has the following structure.

![feedback_motorp.png](figures/feedback_motorp.png)

The closed-loop transfer function for the above with the controller_C_(_s_) simply set equal to 1 can be generated using the MATLAB command|feedback| as shown below.

In [5]:
sys_cl = control.feedback(P_motor,1)

The corresponding unit step response can be generated by adding the aboveand following command to your m-file. The annotations for the peakresponse, settling time, and final value can be added to the plot fromthe right-click menu under *Characteristics*.

<html></p><pre class="codeinput">step(sys_cl,t)</pre></html>

![Figure1.png](figures/Figure1.png)

Examining the above closed-loop step response, the addition of feedbackhas stabilized the system. In fact, the steady-state error appears to bedriven to zero and the overshoot is less than 16%, though the settle timerequirement is not met. The character of the resulting step response isagain indicated by the location of the poles of the system's transferfunction just like the system's stability properties were. The MATLABcommand |pzmap| will plot the poles (and zeros) of a given transferfunction as shown below.

<html></p><pre class="codeinput">pzmap(sys_cl)</pre></html>

![Figure2.png](figures/Figure2.png)

The above plot shows that the closed-loop system has one real pole at-1.45e6 and a pair of complex poles at -29.6+35.3j and -29.6-35.3j asindicated by the locations of the blue |x|'s. The damping and naturalfrequencies associated with these poles can be determined byright-clicking on the associated poles in the resulting plot. Thisinformation can also be determined using the MATLAB command |damp| asshown below.

In [6]:
damp(sys_cl)

Since the one real pole is so much faster than the complex conjugatepoles (its real part is much more negative) its effect on the dynamicresponse of the system will be mimimal. Therefore, the damping ($\zeta$ =0.643) and the natural frequency ($\omega_n$ = 46.1) of the complexconjugate poles will primarily indicate the response of the closed-loopsystem.

Adding the following commands to your m-file will calculate theovershoot and 2% settle time predicted by these poles assuming that theydominate, in other words, that we have a canonical underdamped second-order system.

In [7]:
np.array([Wn,zeta,poles]) = damp(sys_cl)Mp = exp((-zeta(1)*pi)/sqrt(1-zeta(1)**2))Ts = 4/(zeta(1)*Wn(1))

The above results closely match the overshoot and settle time from thestep response plot above which explicitly captured the effect of thethird, non-dominant pole.

Throughout the rest of the pages of this example, different controllers will bedesigned to reduce the settle time of the closed-loop step response tomeet the given 40 millisecond requirement while still meeting the othersystem requirements, including the zero steady-state error in thepresence of a step disturbance.